In [1]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 50.4 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

print("CUDA:", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))

CUDA: True
GPU : Tesla T4


In [4]:
model_name          = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer           = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print("tokenizer ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

tokenizer ready


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit             = True,
    bnb_4bit_quant_type      = "nf4",
    bnb_4bit_compute_dtype   = torch.float16,
    bnb_4bit_use_double_quant= True
)

model                  = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map          = "auto"
)
model.config.use_cache = False
print("model loaded")
print("memory used:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model loaded
memory used: 0.77 GB


In [6]:
lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = ["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [7]:
from google.colab import files
uploaded = files.upload()

Saving val.jsonl to val.jsonl
Saving train.jsonl to train.jsonl


In [8]:
dataset = load_dataset(
    "json",
    data_files={
        "train"     : "/content/train.jsonl",
        "validation": "/content/val.jsonl"
    }
)

def format_prompt(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    }

def tokenize(example):
    return tokenizer(
        format_prompt(example)["text"],
        truncation = True,
        max_length = 256,
        padding    = "max_length"
    )

tokenized = dataset.map(tokenize, remove_columns=dataset["train"].column_names)
print(tokenized)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 300
    })
})


In [9]:
training_args = TrainingArguments(
    output_dir                  = "/content/lora_outputs",
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    learning_rate               = 2e-4,
    num_train_epochs            = 3,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    fp16                        = True,
    gradient_checkpointing      = True,
    optim                       = "paged_adamw_8bit",
    report_to                   = "none"
)
print("training args ready")

training args ready


In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized["train"],
    eval_dataset  = tokenized["validation"],
    data_collator = data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.804998,0.785204
2,0.788428,0.782265
3,0.735986,0.784223


TrainOutput(global_step=900, training_loss=0.7854849836561415, metrics={'train_runtime': 713.6779, 'train_samples_per_second': 5.044, 'train_steps_per_second': 1.261, 'total_flos': 5732896761446400.0, 'train_loss': 0.7854849836561415, 'epoch': 3.0})

In [11]:
adapter_path = "/content/adapters"

trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("adapters saved")

adapters saved


In [ ]:
!zip -r adapters_tinyllama.zip /content/adapters

from google.colab import files
files.download("adapters.zip")

  adding: content/adapters/ (stored 0%)
  adding: content/adapters/tokenizer_config.json (deflated 46%)
  adding: content/adapters/README.md (deflated 66%)
  adding: content/adapters/adapter_model.safetensors (deflated 8%)
  adding: content/adapters/adapter_config.json (deflated 57%)
  adding: content/adapters/chat_template.jinja (deflated 60%)
  adding: content/adapters/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>